In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [2]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="flan_t5_full_answer_logprob_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [3]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [4]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

decoder_start_token_id = model.config.decoder_start_token_id
if decoder_start_token_id is None:
    decoder_start_token_id = tokenizer.pad_token_id

eos_token_id = tokenizer.eos_token_id
assert eos_token_id is not None, "Tokenizer must define eos_token_id for sequence scoring."

candidate_token_ids = {
    "yes": tokenizer("yes", add_special_tokens=False).input_ids + [eos_token_id],
    "no": tokenizer("no", add_special_tokens=False).input_ids + [eos_token_id],
}

print(model_name)
print("decoder_start_token_id:", decoder_start_token_id)
print("eos_token_id:", eos_token_id)
print({k: {"ids": v, "decoded": tokenizer.decode(v)} for k, v in candidate_token_ids.items()})



---[ TableVault Record ]---


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


google/flan-t5-small
decoder_start_token_id: 0
eos_token_id: 1
{'yes': {'ids': [4273, 1], 'decoded': 'yes</s>'}, 'no': {'ids': [150, 1], 'decoded': 'no</s>'}}
---[ TableVault Record ]---



In [5]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_list(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

def build_prompt(s1, s2):
    return (
        "Are these two sentences paraphrases? Answer yes or no.\n"
        f"Sentence 1: {s1}\n"
        f"Sentence 2: {s2}\n"
        "Answer:"
    )

prompts = [build_prompt(a, b) for a, b in zip(sent1, sent2)]

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())
print("sample_prompt:\n", prompts[0])



---[ TableVault Record ]---
Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
sample_prompt:
 Are these two sentences paraphrases? Answer yes or no.
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
Answer:
---[ TableVault Record ]---



In [6]:
def score_candidate(enc, candidate_ids):
    batch_size = enc["input_ids"].shape[0]
    target_ids = torch.tensor(candidate_ids, dtype=torch.long, device=device).unsqueeze(0).expand(batch_size, -1)
    target_len = target_ids.shape[1]

    decoder_input_ids = torch.full(
        (batch_size, target_len),
        fill_value=decoder_start_token_id,
        dtype=torch.long,
        device=device,
    )
    if target_len > 1:
        decoder_input_ids[:, 1:] = target_ids[:, :-1]

    outputs = model(**enc, decoder_input_ids=decoder_input_ids)
    log_probs = torch.log_softmax(outputs.logits, dim=-1)
    token_log_probs = log_probs.gather(dim=-1, index=target_ids.unsqueeze(-1)).squeeze(-1)
    seq_log_probs = token_log_probs.sum(dim=-1)
    return seq_log_probs

batch_size = 32
preds = []
yes_scores_all = []
no_scores_all = []

with torch.inference_mode():
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i + batch_size]

        enc = tokenizer(
            batch_prompts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        yes_scores = score_candidate(enc, candidate_token_ids["yes"])
        no_scores = score_candidate(enc, candidate_token_ids["no"])

        batch_preds = (yes_scores > no_scores).long().cpu().numpy()

        preds.extend(batch_preds.tolist())
        yes_scores_all.extend(yes_scores.cpu().numpy().tolist())
        no_scores_all.extend(no_scores.cpu().numpy().tolist())

y_pred = np.array(preds)
yes_scores_all = np.array(yes_scores_all)
no_scores_all = np.array(no_scores_all)
print("done")



---[ TableVault Record ]---


  0%|          | 0/13 [00:00<?, ?it/s]

done
---[ TableVault Record ]---



In [7]:

vault.create_record_list("flan_prediction_scored", column_names=["prediction", "yes_scores", "no_scores"])

for i in range(len(y_pred)):
    vault.append_record("flan_prediction_scored", 
                        {
                            "prediction": int(y_pred[i]),
                            "yes_scores": float(yes_scores_all[i]),
                            "no_scores": float(no_scores_all[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "flan_prediction_scored is a per-example prediction dataset generated by running google/flan-t5-small on the GLUE MRPC validation set with a yes/no paraphrase prompt. Each record corresponds to one input pair from glue_mrpc_validation and stores the model\u2019s binary prediction and the sequence log-probability scores for the two candidate answers. The fields are: prediction (integer label, where 1 indicates yes/paraphrase and 0 indicates no/not paraphrase), yes_scores (float log-probability of the full answer token sequence for \u201cyes\u201d), and no_scores (float log-probability of the full answer token sequence for \u201cno\u201d). In this workflow, this dataset serves as the scored inference output used to inspect example-level model behavior and to compute downstream summary metrics such as accuracy, F1, and the classification report."
embedding = get_embeddings(description)
vault.create_description("flan_prediction_scored", description, embedding)

properties = {"task": "paraphrase detection", "record_type": "model prediction scores", "source": "glue/mrpc", "derived_from": "glue_mrpc_validation", "split": "validation", "size": "408", "model": "google/flan-t5-small", "model_family": "flan-t5", "scoring_method": "full answer logprob", "prompt_format": "yes/no question answering", "input_type": "sentence pair", "output_fields": "prediction, yes_scores, no_scores", "label_space": "yes/no", "domain": "news"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_prediction_scored", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---



In [8]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))



---[ TableVault Record ]---
{'accuracy': 0.6495098039215687, 'f1': 0.6911447084233261}
                precision    recall  f1-score   support

not_paraphrase       0.47      0.81      0.59       129
    paraphrase       0.87      0.57      0.69       279

      accuracy                           0.65       408
     macro avg       0.67      0.69      0.64       408
  weighted avg       0.74      0.65      0.66       408

---[ TableVault Record ]---



In [9]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "answer:", "yes" if int(y_pred[i]) == 1 else "no")
    print("yes_logprob:", float(yes_scores_all[i]), "no_logprob:", float(no_scores_all[i]))



---[ TableVault Record ]---
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0 answer: no
yes_logprob: -0.8191786408424377 no_logprob: -0.602909505367279
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 answer: no
yes_logprob: -1.6394611597061157 no_logprob: -0.2748245298862457
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 0 answer: no
yes_logprob: -1.0107052326202393 no_l

In [10]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("yes_logprob:", float(yes_scores_all[i]), "no_logprob:", float(no_scores_all[i]))



---[ TableVault Record ]---
num_errors: 143
idx: 0
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0
yes_logprob: -0.8191786408424377 no_logprob: -0.602909505367279
idx: 3
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will decide in October whether to endorse a candidate before the primaries .
true: 1 pred: 0
yes_logprob: -0.821238100528717 no_logprob: -0.6100975871086121
idx: 7
sentence1: This integrates with Rational PurifyPlus and allows developers to work in supported versions of Java , Visual C # and Visual Basic .NET.
sentence2: IBM said the Rational products were also integrated with Rational PurifyPlus , which allows developers to work in Java , Visual C # and VisualBasic .Net.
true: 1 pred: 0
yes_logprob: -0.9792486429214478 no

In [11]:

vault.create_record_list("flan_t5_full_answer_logprob_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("flan_t5_full_answer_logprob_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "flan_prediction_scored": [0, len(ds)]
                    })

summary

description = "flan_t5_full_answer_logprob_mrpc_summary is an experiment-level summary dataset for evaluating google/flan-t5-small on the GLUE MRPC validation split using prompt-based yes/no paraphrase prediction with full-answer log-probability scoring. It contains a single record with three fields: accuracy (float), f1 (float), and classification_report (string from sklearn with per-class precision, recall, f1, and support for not_paraphrase and paraphrase). This dataset is produced after generating example-level predictions in flan_prediction_scored and comparing them against the ground-truth MRPC labels. Its role in the workflow is to store the aggregate evaluation results for the entire validation run, linking the underlying input dataset (glue_mrpc_validation) and the per-example prediction dataset (flan_prediction_scored)."
embedding = get_embeddings(description)
vault.create_description("flan_t5_full_answer_logprob_mrpc_summary", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "evaluation summary", "source": "glue/mrpc", "dataset": "MRPC", "split": "validation", "size": "408", "model": "google/flan-t5-small", "inference_method": "full answer log probability scoring", "label_space": "yes/no", "input_format": "sentence pair prompt", "metrics": "accuracy,f1,classification_report", "language": "English"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_t5_full_answer_logprob_mrpc_summary", cat, embedding, prop)




---[ TableVault Record ]---
---[ TableVault Record ]---



In [12]:
description = "This notebook runs a prompt-based paraphrase detection experiment on the GLUE MRPC validation set using google/flan-t5-small. For each sentence pair, it builds an instruction prompt asking whether the two sentences are paraphrases and scores the full decoder log probability of the candidate answers \u201cyes\u201d and \u201cno\u201d (including EOS). The predicted label is chosen by comparing these two sequence log probabilities.\n\nThe workflow loads MRPC validation data from TableVault, tokenizes prompts, performs batched inference with FLAN-T5, records per-example predictions and yes/no scores, and evaluates performance with accuracy, F1, and a classification report. It also prints sample predictions and error cases for qualitative inspection.\n\nResults are written back to TableVault as a scored prediction record list and a summary record list, with dataset lineage links and embedding-based descriptions/metadata for both the outputs and the overall notebook process." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("flan_t5_full_answer_logprob_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "dataset": "glue/mrpc", "split": "validation", "model": "google/flan-t5-small", "model_family": "flan-t5", "approach": "prompt-based binary classification via seq2seq log-probability scoring", "label_space": "yes/no", "scoring_method": "full answer log probability comparison", "frameworks": "transformers, pytorch, datasets, scikit-learn", "evaluation": "accuracy, f1-score, classification report", "artifact_outputs": "per-example predictions and summary metrics", "tracking": "tablevault", "embedding_model": "text-embedding-3-large"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_t5_full_answer_logprob_mrpc", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---

